# traceability_export - PR calculation traceability Excel (mode B: with Excel formulas)

Generates one Excel workbook per plant and period with **real Excel formulas** in the
calculated sheets, so the client can modify thresholds or readings and see the delta
recalculate live.

**Workbook sheets:**
1. `00_Summary` - main KPIs (with formulas pointing to `05_Interval_Comparison`)
2. `01_Contract_Config` - applied thresholds (source of the formula references)
3. `02_SCADA_Raw` - SCADA readings as-is (values)
4. `03_SCADA_Processed` - raw + formulas (POA filt, ALB, c1-c4, all_valid)
5. `04_PVsyst_Simulation` - simulated reference (values)
6. `05_Interval_Comparison` - formulas: Measured/Expected/Guaranteed Energy, Delta
7. `06_Daily_Summary` - daily SUMIFS/SUMPRODUCT formulas over `05_Interval_Comparison`
8. `07_Chart` - charts (daily valid energy, daily EPI, hourly profile)
9. `08_Outliers` - values (filtered subset of `03`)

The formulas use `LET` and `SUMPRODUCT`, compatible with Excel 365 / 2021+.

**Default period:** last 3 months (`num_months = 3`, configurable in the first cell). Only applies if `period_start`/`period_end` are empty.

In [1]:
# Parameters
plant_codes  = ""       # one or more plants (list, JSON, or comma-separated). Empty = all
period_start = ""          # ISO YYYY-MM-DD. If set, overrides last_months_only.
period_end   = ""          # ISO YYYY-MM-DD. Empty = latest available.

# Filter to the last N months. Only applies if period_start and period_end are empty.
last_months_only = True
num_months = 3

# If True, ignore the 'traceability_excel' column in 01_Plants and export ALL
# plants listed in plant_codes. If False, export only those with
# traceability_excel = TRUE in plants_config.
force_export = False


In [2]:

# --- Azure AD registered app: "PVSyst1_EPI" ---
client_id  = "58f369e5-0dfd-4435-802b-a90a6786bf22"
tenant_id  = "943fa85b-068d-404a-bacb-5f0baf8d31a8"

# Key Vault holding the client_secret
key_vault_url             = "https://apicredentialsenerland.vault.azure.net/"
secret_client_secret_name = "PVSyst1EPISecret"

# SharePoint destination
sp_site_path     = "enerlandgroup.sharepoint.com:/sites/Pvsyst1"
sp_drive_name    = "Documentos"
sp_traceability_path = "TRACEABILITY"

# Local audit
local_audit_root = "traceability_audit"


In [3]:
import io
import json
import requests
import pandas as pd
import numpy as np
import notebookutils
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import quote

LAKEHOUSE_FILES = '/lakehouse/default/Files'
ABFS_BASE = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"


def _read_sheet(sheet, table=None, header=0):
    """Reads a STRUCTURED TABLE from the config Excel: 1) by table NAME
    (case-insensitive, on any sheet); 2) otherwise the (first) table on the given
    sheet; 3) as a last resort, the whole sheet. Robust to renames."""
    local = '/tmp/plants_config_trace.xlsx'
    if not Path(local).exists():
        abfs_path = ABFS_BASE + '/Files/Raw PVSyst1/CONFIG/plants_config.xlsx'
        row = spark.read.format('binaryFile').load(abfs_path).first()
        with open(local, 'wb') as _f:
            _f.write(row['content'])
    import openpyxl
    wb = openpyxl.load_workbook(local, data_only=True)
    def _df(ws, ref):
        _d = [[c.value for c in _r] for _r in ws[ref]]
        return pd.DataFrame(_d[1:], columns=_d[0]) if _d else pd.DataFrame()
    if table:
        for ws in wb.worksheets:
            _tm = {k.lower(): k for k in ws.tables}
            if table.lower() in _tm:
                return _df(ws, ws.tables[_tm[table.lower()]].ref)
    if sheet in wb.sheetnames:
        ws = wb[sheet]
        if ws.tables:
            return _df(ws, ws.tables[next(iter(ws.tables))].ref)
    return pd.read_excel(local, sheet_name=sheet, header=header)
def _getf(row, key, default):
    if key not in row.index:
        return default
    v = row[key]
    try:
        return float(v) if v is not None and not pd.isna(v) else default
    except (ValueError, TypeError):
        return default


def _getf_v(v, default):
    try:
        return float(v) if v is not None and not pd.isna(v) else default
    except (ValueError, TypeError):
        return default


def _bool(v, default=False):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return default
    s = str(v).strip().upper()
    return s in ('TRUE', 'VERDADERO', '1', 'SÍ', 'SI', 'YES', 'Y')


def _parse_selected_criteria(plant_code):
    """04_Criteria_Selection (table T04: plant_code, c1..c9). c1-c7 numeric (-1 off), c8-c9 bool."""
    sel = {f'c{i}': (0.0 if i <= 7 else True) for i in range(1, 10)}
    try:
        sc = _read_sheet('04_Criteria_Selection', 'T04_criteria_selection')
        sc.columns = [str(c).strip().lower() for c in sc.columns]
        sub = sc[sc['plant_code'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for n in range(1, 10):
                ck = f'c{n}'
                if ck in sc.columns:
                    sel[ck] = _getf_v(row[ck], 0.0) if n <= 7 else _bool(row[ck], True)
    except Exception:
        pass
    return sel
def _parse_monthly_irr(plant_code):
    """07_Irrandiance_Thresholds (table T07: plant_code, GHI_m1..POA_m12)."""
    poa, ghi = {}, {}
    try:
        df = _read_sheet('07_Irrandiance_Thresholds', 'T07_irradiance_threshold')
        df.columns = [str(c).strip().upper() for c in df.columns]
        sub = df[df['PLANT_CODE'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for m in range(1, 13):
                if f'POA_M{m}' in df.columns:
                    poa[m] = _getf_v(row[f'POA_M{m}'], 0.0)
                if f'GHI_M{m}' in df.columns:
                    ghi[m] = _getf_v(row[f'GHI_M{m}'], 0.0)
    except Exception:
        pass
    return poa, ghi
def _load_sensor_outliers(plant_code):
    """05_Sensor_Outliers -> per family, the 3-method cleaning params:
        iqr_k       -> Method 1 (IQR/Tukey k; 0 = off)
        dev_rounds  -> Method 2 (up to 3 median-relative % rounds; each 0/empty = skipped)
        max_diff    -> Method 3 (absolute deviation from the median, W/m2; 0 = off)
    Accepts the standardized column names (<FAM>_iqr_k/_dev1/_dev2/_dev3/_maxdiff)
    and the older inconsistent names as a fallback. Cleaning itself runs in
    scada_proc; here it only feeds the 01_Contract_Config reference sheet."""
    fams  = {'POA': 'POA', 'GHI': 'GHI', 'DHI': 'DHI', 'REF': 'REF', 'T_AMB': 'TAMB'}
    alias = {
        'iqr_k':   ['{p}_iqr_k'],
        'dev1':    ['{p}_dev1', '{p}_max_dev_1', '{p}_max_dev', '{p}_max_dev4'],
        'dev2':    ['{p}_dev2', '{p}_max_dev_2', '{p}_max_dev2'],
        'dev3':    ['{p}_dev3', '{p}_max_dev_3', '{p}_max_dev3'],
        'maxdiff': ['{p}_maxdiff', '{p}_max_diff'],
    }
    srow = None
    try:
        so  = _read_sheet('05_Sensor_Outliers', 'T05_sensor_outliers')
        sub = so[so['plant_code'] == plant_code]
        if not sub.empty:
            srow = sub.iloc[0]
    except Exception:
        srow = None

    def _pick(pre, key):
        if srow is None:
            return 0.0
        for name in alias[key]:
            col = name.format(p=pre)
            if col in srow.index:
                return _getf(srow, col, 0.0)
        return 0.0

    out = {}
    for fam, pre in fams.items():
        out[fam] = {
            'iqr_k':      _pick(pre, 'iqr_k'),
            'dev_rounds': [_pick(pre, 'dev1'), _pick(pre, 'dev2'), _pick(pre, 'dev3')],
            'max_diff':   _pick(pre, 'maxdiff'),
        }
    return out


def _load_albedo_mapping(plant_code):
    """09_Albedo_Mapping: for each albedo measure, which direct (denominator) and
    reflected (numerator) sensors compute it. COMMON is the base; the plant's rows
    add or override by albedo_measure (same convention as 02_Sensors_Mapping).
    Returns an ordered dict {measure: (direct_sensor, reflected_sensor)}."""
    try:
        m = _read_sheet('09_Albedo_Mapping', 'Tabla9')
    except Exception:
        return {}
    m.columns = [str(c).strip().lower() for c in m.columns]
    if 'plant_code' not in m.columns:
        return {}
    def _cell(row, *names):
        for n in names:
            if n in row.index and pd.notna(row[n]):
                return str(row[n]).strip()
        return None
    out = {}
    for pc in ('COMMON', plant_code):
        for _, r in m[m['plant_code'].astype(str).str.strip() == pc].iterrows():
            meas   = _cell(r, 'albedo_measure')
            direct = _cell(r, 'direct_sensor')
            refl   = _cell(r, 'reflected_sensor', 'reflected_sentor')
            if meas and direct and refl:
                out[meas] = (direct, refl)
    return out


def _epi_factors(plant_code):
    """03_EPI_Factors: base row (blank period) + optional period rows. Returns
    (base, periods): base={epi,deg,avail}; periods=[{start,end,epi,deg,avail}]
    (Timestamps). Rows with blank period_start/period_end are the base factors."""
    base = {'epi': 0.98, 'deg': 1.0, 'avail': 1.0}
    periods = []
    try:
        fdf = _read_sheet('03_EPI_Factors', 'T03_criteria_config')
        fdf.columns = [str(c).strip().lower() for c in fdf.columns]
        sub = fdf[fdf['plant_code'].astype(str).str.strip() == plant_code]
        for _, r in sub.iterrows():
            fac = {'epi': _getf(r, 'guaranteed_epi', base['epi']),
                   'deg': _getf(r, 'degradation_factor', 1.0),
                   'avail': _getf(r, 'availability_factor', 1.0)}
            ps = pd.to_datetime(r.get('period_start'), errors='coerce')
            pe = pd.to_datetime(r.get('period_end'), errors='coerce')
            if pd.isna(ps) and pd.isna(pe):
                base = fac
            else:
                periods.append({'start': ps, 'end': pe, **fac})
    except Exception:
        pass
    return base, periods


def load_plant_config(plant_code):
    """Reads 01_Plants + 03_EPI_Factors + 04_Criteria_Selection + 07 + 05 + 09.
    Returns the canonical config for traceability."""
    plants = _read_sheet('01_Plants', 'T01_plants')

    prow = plants[plants['plant_code'] == plant_code]
    if prow.empty:
        raise ValueError(f'{plant_code} not found in 01_Plants')
    prow = prow.iloc[0]

    epi_base, epi_periods = _epi_factors(plant_code)

    sel = _parse_selected_criteria(plant_code)
    poa_m, ghi_m = _parse_monthly_irr(plant_code)

    # Active criteria: c1-c7 active if value >= 0; c8-c9 if True.
    active = set()
    for n in range(1, 8):
        if sel.get(f'c{n}', 0.0) is not None and sel.get(f'c{n}', 0.0) >= 0:
            active.add(f'c{n}')
    for n in (8, 9):
        if sel.get(f'c{n}', True):
            active.add(f'c{n}')

    return {
        'plant_code':           plant_code,
        'plant_name':           str(prow['plant_name']),
        'ac_capacity_kw':       _getf(prow, 'ac_capacity_kw', 0.0),
        'dc_capacity_kw':       _getf(prow, 'dc_capacity_kw', 0.0),
        'pvsyst_file':          str(prow.get('pvsyst_file', '')),
        'timezone':             str(prow.get('timezone', '')),

        # Base EPI factors (03_EPI_Factors, blank-period row); per-period overrides below.
        'guaranteed_epi':       epi_base['epi'],
        'degradation_factor':   epi_base['deg'],
        'availability_factor':  epi_base['avail'],
        'epi_periods':          epi_periods,
        'albedo_mapping':       _load_albedo_mapping(plant_code),
        # Sensor outlier cleaning params per family (05_Sensor_Outliers)
        'sensor_clean':         _load_sensor_outliers(plant_code),

        # --- Criteria (04_Criteria_Selection) ---
        'selected_criteria':    sel,
        'active_criteria':      active,
        # per-criterion thresholds (value from 04)
        'c1_thr':  sel.get('c1', 0.0),
        'c2_thr':  sel.get('c2', 0.0),
        'c3_thr':  sel.get('c3', 0.0),
        'c4_thr':  sel.get('c4', 0.0),
        'c5_hours': sel.get('c5', 0.0),
        'c6_hours': sel.get('c6', 0.0),
        'c7_setpoint': sel.get('c7', 0.0),
        'c8_wind_alarm': bool(sel.get('c8', False)),
        'c9_pf_rango':   bool(sel.get('c9', False)),

        # --- Monthly minimum irradiance curves (07) ---
        'monthly_thr_poa':      poa_m,
        'monthly_thr_ghi':      ghi_m,

        'aggregate_by_hour':    _bool(prow.get('aggregate_by_hour') if 'aggregate_by_hour' in prow.index else None, False),
        'is_energy':            _bool(prow.get('is_energy') if 'is_energy' in prow.index else None, False),
    }
# ---------------------------------------------------------
# Time-shift T08 — se aplica al SCADA, ANTES de agrupar
# ---------------------------------------------------------
# `time_shift` de T08_time_shifts son los minutos que la conversion PVsyst
# adelanto el meteo antes de simular (auto_run.bat + scripts/timeshift.py en el
# PC de PVsyst). PVsyst devuelve su CSV en la rejilla horaria limpia (:00), asi
# que el desfase NO se toca en proc_pvsyst: se aplica aqui sobre las fechas del
# SCADA, que es la serie fina, y ANTES de agregar a hora.
#
# Se redondea a la resolucion NATIVA del raw SCADA (15 min en estas plantas):
# con un desfase que no cae en esa rejilla, el resample posterior mandaria unas
# filas al bucket de arriba y otras al de abajo. Redondeando a la rejilla el
# agrupado es determinista y el desfase sobrevive entero al bucket horario (los
# 75 min de PL1 siguen siendo 75, no 60).
#
# Al ser multiplo de la rejilla, el SCADA desplazado SIGUE siendo una rejilla de
# 15 min y cada bucket horario conserva sus 4 intervalos: solo quedan buckets
# parciales en los bordes del periodo y en la costura del cambio de hora.
# Lo que si deja un desfase que no es multiplo de 60 (p.ej. los 75 de PL1) es que
# uno de los 4 cuartos de cada hora de PVsyst cae en el bucket de al lado: se
# alinea mejor la hora solar a cambio de mezclar un cuarto con la hora vecina.
#
# Semantica de periodos (la misma que documenta T08):
#   - Lo que no cubre ningun periodo queda a 0.
#   - end_date es INCLUSIVO, y las costuras caen a las DST_SEAM_HOUR del dia
#     indicado: el reloj en la UE cambia a las 02:00/03:00 locales, no a
#     medianoche, asi que cortar a dia completo desalinearia la madrugada.
#   - Puede haber VARIOS periodos por planta; si se solapan gana el de MAS
#     ABAJO en la tabla (las reglas se aplican en orden).
#   - Un 0 explicito es una regla valida: anula un periodo anterior solapado.
#   - Fecha vacia = sin limite por ese lado.
DST_SEAM_HOUR = 2


def t08_rules(plant_code):
    """T08_time_shifts -> [(start, end, minutos)] de la planta, EN EL ORDEN DE LA TABLA."""
    rules = []
    try:
        df = _read_sheet('08_Time_Shifts', 'T08_time_shifts')
        df.columns = [str(c).strip().lower() for c in df.columns]
        sub = df[df['plant_code'].astype(str).str.strip().str.upper()
                 == str(plant_code).strip().upper()]
        for _, r in sub.iterrows():
            try:
                raw = float(r.get('time_shift'))
            except (TypeError, ValueError):
                continue          # celda vacia o no numerica: no es una regla
            if pd.isna(raw):
                continue
            start = pd.to_datetime(r.get('start_date'), errors='coerce')
            end   = pd.to_datetime(r.get('end_date'),   errors='coerce')
            rules.append((None if pd.isna(start) else start,
                          None if pd.isna(end)   else end,
                          raw))
    except Exception as e:
        print(f'  [time_shift] no se pudieron leer reglas T08 para {plant_code}: {e}')
    return rules


def scada_time_shift(dates, plant_code, scada_res):
    """Minutos de desfase de T08, con el signo de T08 (los que se le suman a PVsyst).

    Aqui se RESTAN a las fechas del SCADA para agrupar igual que pr_runner; la
    columna A del Excel se queda en hora real y son las busquedas contra PVsyst
    las que llevan el -shift.

    `scada_res` es la resolucion nativa del raw SCADA y define la rejilla a la
    que se redondea el desfase. Devuelve una Serie int64 alineada con `dates`
    (0 en las filas que no cubre ninguna regla).
    """
    step = max(1, int(round(pd.Timedelta(scada_res).total_seconds() / 60)))
    out  = pd.Series(0, index=dates.index, dtype='int64')
    seam = pd.Timedelta(hours=DST_SEAM_HOUR)
    for start, end, raw in t08_rules(plant_code):
        mins = int(round(raw / step)) * step
        mask = pd.Series(True, index=dates.index)
        if start is not None:
            mask &= dates > pd.Timestamp(start).normalize() + seam
        if end is not None:
            mask &= dates <= pd.Timestamp(end).normalize() + seam
        out.loc[mask] = mins
    return out


In [4]:
# Microsoft Graph helpers (same flow as meteo_export / pr_runner)
_GRAPH_BASE = "https://graph.microsoft.com/v1.0"
_TIMEOUT    = 60


def get_graph_token():
    client_secret = notebookutils.credentials.getSecret(key_vault_url, secret_client_secret_name)
    r = requests.post(
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
        data={'client_id': client_id, 'scope': 'https://graph.microsoft.com/.default',
              'client_secret': client_secret, 'grant_type': 'client_credentials'},
        timeout=_TIMEOUT,
    )
    if not r.ok:
        raise RuntimeError(f"OAuth fallo {r.status_code}: {r.text[:400]}")
    return r.json()['access_token']


def _gget(token, path):
    r = requests.get(f"{_GRAPH_BASE}/{path.lstrip('/')}",
                     headers={'Authorization': f'Bearer {token}'},
                     timeout=_TIMEOUT)
    if not r.ok:
        raise RuntimeError(f"Graph GET {path} -> {r.status_code}: {r.text[:400]}")
    return r.json()


def resolve_site_and_drive(token):
    site = _gget(token, f"sites/{sp_site_path}")
    site_id = site['id']
    if sp_drive_name:
        drives = _gget(token, f"sites/{site_id}/drives").get('value', [])
        drive = next((d for d in drives if d.get('name') == sp_drive_name), None)
        if drive is None:
            raise ValueError(f"Biblioteca '{sp_drive_name}' no encontrada. Disponibles: {[d.get('name') for d in drives]}")
        drive_id = drive['id']
    else:
        drive_id = _gget(token, f"sites/{site_id}/drive")['id']
    return site_id, drive_id


def upload_to_sharepoint(token, site_id, drive_id, dest_relpath, content_bytes):
    safe_path = "/".join(quote(seg, safe='') for seg in dest_relpath.strip('/').split('/'))
    url = f"{_GRAPH_BASE}/sites/{site_id}/drives/{drive_id}/root:/{safe_path}:/content"
    r = requests.put(
        url,
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/octet-stream'},
        data=content_bytes,
        timeout=_TIMEOUT * 4,
    )
    if not r.ok:
        raise RuntimeError(f"Graph PUT {dest_relpath} -> {r.status_code}: {r.text[:400]}")
    j = r.json()
    return {'name': j.get('name'), 'webUrl': j.get('webUrl'), 'size': j.get('size')}


In [5]:
# ---------------------------------------------------------------------------
# Excel builder (openpyxl) - hybrid mode:
#   - Sheets 02/03/04/08: precomputed VALUES from the Delta tables (no formulas).
#   - Sheets 05/06/07/00: Excel FORMULAS that recompute when EPI/Deg/Avail change.
#   - Sheet 01: configuration (source of references for the formulas).
# ---------------------------------------------------------------------------
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# --- Styles ----------------------------------------------------------------
_HDR_FILL    = PatternFill('solid', start_color='1F4E79', end_color='1F4E79')
_HDR_FONT    = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
_HDR_ALIGN   = Alignment(horizontal='center', vertical='center', wrap_text=True)
_NOTE_FILL   = PatternFill('solid', start_color='FFF2CC', end_color='FFF2CC')
_NOTE_FONT   = Font(name='Calibri', size=10, italic=True, color='3D3D3D')
_FORMULA_FILL= PatternFill('solid', start_color='F2F2F2', end_color='F2F2F2')
_KPI_FILL_OK = PatternFill('solid', start_color='E2EFDA', end_color='E2EFDA')
_KPI_FILL_KO = PatternFill('solid', start_color='FCE4D6', end_color='FCE4D6')
_KPI_FONT    = Font(name='Calibri', size=12, bold=True)
_THIN        = Side(style='thin', color='BFBFBF')
_BORDER      = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)

# Status / anomaly colors
_NULL_FILL      = PatternFill('solid', start_color='D9D9D9', end_color='D9D9D9')  # grey
_ANOMALY_FILL   = PatternFill('solid', start_color='F4B6B6', end_color='F4B6B6')  # light red
_OUTLIER_FILL   = PatternFill('solid', start_color='F8CBAD', end_color='F8CBAD')  # salmon
_DISCARDED_FILL = PatternFill('solid', start_color='FFE699', end_color='FFE699')  # amber
_STATUS_FONT    = Font(name='Calibri', size=10, italic=True, bold=True, color='843C0C')
_STATUS_ALIGN   = Alignment(horizontal='center', vertical='center')

# --- Absolute references to 01_Contract_Config (hardcoded rows) ------------
CFG = "'01_Contract_Config'"
# FIXED row layout of 01_Contract_Config (see _config_layout). Do NOT reorder.
REF_EPI_GUAR     = f"{CFG}!$B$8"   # guaranteed_epi
REF_DEGRADATION  = f"{CFG}!$B$9"   # degradation_factor
REF_AVAILABILITY = f"{CFG}!$B$10"  # availability_factor
REF_C1_THR       = f"{CFG}!$B$16"  # c1 POA_avg threshold (0 = non-null only)
REF_C2_THR       = f"{CFG}!$B$17"  # c2 GHI_avg threshold
REF_C3_THR       = f"{CFG}!$B$18"  # c3 DHI_avg threshold
REF_C4_THR       = f"{CFG}!$B$19"  # c4 REF_avg threshold
REF_C5_HOURS     = f"{CFG}!$B$20"  # c5 hours/day POA
REF_C6_HOURS     = f"{CFG}!$B$21"  # c6 hours/day GHI
REF_C7_SETPT     = f"{CFG}!$B$22"  # c7 setpoint threshold kW
# Outlier thresholds and dt_h are displayed in 01_Contract_Config for reference
# only; the sensor cleaning is computed in scada_proc, not by live Excel formulas.


def _explain(ws, text, ncols=8, row=1):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncols)
    c = ws.cell(row=row, column=1, value=text)
    c.fill = _NOTE_FILL
    c.font = _NOTE_FONT
    c.alignment = Alignment(wrap_text=True, vertical='top', horizontal='left')
    n_lines = max(text.count('\n') + 1, 3)
    ws.row_dimensions[row].height = 18 * n_lines


def _hdr(ws, cols, row):
    for i, c in enumerate(cols, 1):
        cell = ws.cell(row=row, column=i, value=c)
        cell.fill = _HDR_FILL
        cell.font = _HDR_FONT
        cell.alignment = _HDR_ALIGN
        cell.border = _BORDER
    ws.row_dimensions[row].height = 28


def _mark_status(cell, label):
    upper = (label or '').upper()
    fill_map = {
        'NULL':         _NULL_FILL,
        'MISSING':      _NULL_FILL,
        'ANOMALY':      _ANOMALY_FILL,
        'OUT_OF_RANGE': _ANOMALY_FILL,
        'DEAD':         _ANOMALY_FILL,
        'ABRUPT':       _ANOMALY_FILL,
        'OUTLIER':      _OUTLIER_FILL,
        'DISCARDED':    _DISCARDED_FILL,
    }
    fill = fill_map.get(upper)
    if fill:
        cell.fill = fill
    cell.font = _STATUS_FONT
    cell.alignment = _STATUS_ALIGN


def _apply_status_cf(ws, top_left, bottom_right):
    """Applies conditional formatting to a range: colors the cells that contain
    exactly 'NULL'/'MISSING'/'ANOMALY'/.../'OUTLIER'/'DISCARDED'.
    """
    from openpyxl.formatting.rule import CellIsRule
    rng = f'{top_left}:{bottom_right}'
    rules = [
        ('NULL',         _NULL_FILL),
        ('MISSING',      _NULL_FILL),
        ('ANOMALY',      _ANOMALY_FILL),
        ('OUT_OF_RANGE', _ANOMALY_FILL),
        ('DEAD',         _ANOMALY_FILL),
        ('ABRUPT',       _ANOMALY_FILL),
        ('OUTLIER',      _OUTLIER_FILL),
        ('DISCARDED',    _DISCARDED_FILL),
    ]
    for label, fill in rules:
        ws.conditional_formatting.add(rng,
            CellIsRule(operator='equal', formula=[f'"{label}"'], fill=fill, font=_STATUS_FONT))


def _put_val(ws, row, col, v, fmt=None):
    """Writes a value into a cell.
       - Empty string ('') -> leaves the cell EMPTY (unmarked). Used for visual
         separators in 01_Contract_Config.
       - None / NaN / Inf -> writes 'NULL' with a grey background.
       - Known status strings ('OUTLIER', 'DISCARDED', 'OUT_OF_RANGE', etc.) ->
         colored via _mark_status.
       - 'ok' -> soft grey text (no highlight).
       - Otherwise -> value as-is with optional formatting.
    """
    # Empty string -> leave the cell blank, unmarked
    if isinstance(v, str) and not v.strip():
        return

    # None / NaN / Inf -> colored NULL
    is_null = v is None or (isinstance(v, float) and not np.isfinite(v))
    try:
        if not is_null and pd.isna(v):
            is_null = True
    except (TypeError, ValueError):
        pass

    if is_null:
        cell = ws.cell(row=row, column=col, value='NULL')
        _mark_status(cell, 'NULL')
        return

    if isinstance(v, (pd.Timestamp, datetime)):
        cell = ws.cell(row=row, column=col, value=v.to_pydatetime() if hasattr(v, 'to_pydatetime') else v)
        cell.number_format = fmt or 'yyyy-mm-dd hh:mm'
        return

    if isinstance(v, bool):
        ws.cell(row=row, column=col, value=int(v))
        return

    if isinstance(v, (int, np.integer)):
        cell = ws.cell(row=row, column=col, value=int(v))
        if fmt:
            cell.number_format = fmt
        return

    if isinstance(v, (float, np.floating)):
        cell = ws.cell(row=row, column=col, value=float(v))
        if fmt:
            cell.number_format = fmt
        return

    s = str(v)
    cell = ws.cell(row=row, column=col, value=s)
    upper = s.upper()
    if upper in ('NULL', 'OUTLIER', 'DISCARDED', 'ANOMALY',
                 'OUT_OF_RANGE', 'MISSING', 'DEAD', 'ABRUPT'):
        _mark_status(cell, upper)
    elif upper == 'OK':
        # Healthy sensor: soft grey italic text (no highlight)
        cell.font = Font(name='Calibri', size=10, color='6F6F6F', italic=True)
        cell.alignment = _STATUS_ALIGN



def _is_null_value(v):
    """True if v is None / NaN / Inf / pd.NA."""
    if v is None:
        return True
    if isinstance(v, float) and not np.isfinite(v):
        return True
    try:
        if pd.isna(v):
            return True
    except (TypeError, ValueError):
        pass
    return False


def _flag_to_label(flag_v):
    """Maps a *_flag column value to its display label.
    Returns None if the flag indicates the sensor is ok / empty.
    """
    if flag_v is None:
        return None
    try:
        if pd.isna(flag_v):
            return None
    except (TypeError, ValueError):
        pass
    s = str(flag_v).strip().lower()
    if s in ('', 'ok', 'nan', 'none'):
        return None
    return s.upper()


def _put_raw(ws, row, col, raw_v, flag_v=None, fmt=None):
    """For raw sensor columns: if the flag indicates an anomaly, writes the
    flag label (ANOMALY_X) instead of NULL.
    """
    if _is_null_value(raw_v):
        label = _flag_to_label(flag_v)
        if label:
            cell = ws.cell(row=row, column=col, value=label)
            _mark_status(cell, label)
        else:
            cell = ws.cell(row=row, column=col, value='NULL')
            _mark_status(cell, 'NULL')
        return
    _put_val(ws, row, col, raw_v, fmt=fmt)


def _put_filt(ws, row, col, filt_v, raw_v, flag_v=None, fmt=None):
    """For <sensor>_filt columns:
       - If filt is NaN and the flag indicates an anomaly -> flag label.
       - If filt is NaN and raw is also NaN -> NULL (sensor did not report).
       - If filt is NaN but raw was numeric -> DISCARDED (cross-sensor filter).
    """
    if _is_null_value(filt_v):
        label = _flag_to_label(flag_v)
        if label:
            cell = ws.cell(row=row, column=col, value=label)
            _mark_status(cell, label)
        elif _is_null_value(raw_v):
            cell = ws.cell(row=row, column=col, value='NULL')
            _mark_status(cell, 'NULL')
        else:
            cell = ws.cell(row=row, column=col, value='DISCARDED')
            _mark_status(cell, 'DISCARDED')
        return
    _put_val(ws, row, col, filt_v, fmt=fmt)


def _put_avg_filt(ws, row, col, avg_v, family_raw_values, fmt=None):
    """For <FAMILY>_avg_filt columns: if NaN, distinguishes NULL (all sensors
    in the family were NaN) vs DISCARDED (at least one sensor was numeric but the
    filter dropped it).
    """
    if _is_null_value(avg_v):
        all_null = all(_is_null_value(v) for v in family_raw_values)
        label = 'NULL' if all_null else 'DISCARDED'
        cell = ws.cell(row=row, column=col, value=label)
        _mark_status(cell, label)
        return
    _put_val(ws, row, col, avg_v, fmt=fmt)


def _autosize(ws, sample_rows=200, min_w=10, max_w=38):
    for col_cells in ws.columns:
        first = next((c for c in col_cells if hasattr(c, 'column_letter')), None)
        if first is None:
            continue
        letter = first.column_letter
        max_len = min_w
        for cell in list(col_cells)[:sample_rows]:
            try:
                if cell.value is None:
                    continue
                s = str(cell.value)
                if len(s) > max_len:
                    max_len = len(s)
            except Exception:
                pass
        ws.column_dimensions[letter].width = min(max_w, max_len + 2)


def _add_table(ws, name, header_row, last_data_row, n_cols):
    """Turns a range (header_row..last_data_row) x (1..n_cols) into an Excel Table.
    The 'name' is the identifier for structured references (e.g. tbl_proc).
    """
    if last_data_row < header_row or n_cols < 1:
        return
    ref = f"A{header_row}:{get_column_letter(n_cols)}{last_data_row}"
    tab = Table(displayName=name, ref=ref)
    tab.tableStyleInfo = TableStyleInfo(name="TableStyleLight9",
                                        showFirstColumn=False, showLastColumn=False,
                                        showRowStripes=True, showColumnStripes=False)
    ws.add_table(tab)



def _config_layout(cfg, dt_h):
    """Fixed row layout of 01_Contract_Config. The REF_* constants depend on these
    rows - DO NOT REORDER. Rows 16-22 = criteria thresholds (04_Criteria_Selection)."""
    sel = cfg.get('selected_criteria', {})
    rows = [
        ('--- Plant ---', ''),                                              # 3
        ('plant_code',           cfg['plant_code']),                         # 4
        ('plant_name',           cfg['plant_name']),                         # 5
        ('ac_capacity_kw',       cfg['ac_capacity_kw']),                     # 6
        ('dc_capacity_kw',       cfg['dc_capacity_kw']),                     # 7
        ('guaranteed_epi',       cfg['guaranteed_epi']),                     # 8  -> REF_EPI_GUAR
        ('degradation_factor',   cfg.get('degradation_factor', 1.0)),        # 9  -> REF_DEGRADATION
        ('availability_factor',  cfg.get('availability_factor', 1.0)),       # 10 -> REF_AVAILABILITY
        ('energy_ref_sensor',    'E_GRID'),                                  # 11
        ('pvsyst_file',          cfg.get('pvsyst_file', '')),                # 12
        ('timezone',             cfg.get('timezone', '')),                   # 13
        ('', ''),                                                            # 14
        ('--- Criteria (04; -1=disabled | c8-c9: TRUE/FALSE) ---', ''),  # 15
        ('c1 POA_avg threshold (0=non-null)', sel.get('c1', 0.0)),               # 16 -> REF_C1_THR
        ('c2 GHI_avg threshold (0=non-null)', sel.get('c2', 0.0)),               # 17 -> REF_C2_THR
        ('c3 DHI_avg threshold (0=non-null)', sel.get('c3', 0.0)),               # 18 -> REF_C3_THR
        ('c4 REF_avg threshold (0=non-null)', sel.get('c4', 0.0)),               # 19 -> REF_C4_THR
        ('c5 hours/day POA',     sel.get('c5', 0.0)),                        # 20 -> REF_C5_HOURS
        ('c6 hours/day GHI',     sel.get('c6', 0.0)),                        # 21 -> REF_C6_HOURS
        ('c7 setpoint min kW',   sel.get('c7', 0.0)),                              # 22 -> REF_C7_SETPT
        ('c8 Wind Alarm active', bool(sel.get('c8', False))),                      # 23 (bool, no formula)
        ('c9 PF out of range', bool(sel.get('c9', False))),                      # 24 (bool, no formula)
    ]
    # --- Sensor outlier cleaning (05_Sensor_Outliers): only ACTIVE methods shown.
    # Computed in scada_proc; shown here for reference. IQR = Tukey k; % dev = median
    # relative rounds; abs = absolute deviation from the median (W/m2). ---
    rows.append(('--- Sensor outlier cleaning (05_Sensor_Outliers) ---', ''))
    _clean = cfg.get('sensor_clean', {})
    _any_clean = False
    for _fam, _lbl in [('POA', 'POA'), ('GHI', 'GHI'), ('DHI', 'DHI'), ('REF', 'REF'), ('T_AMB', 'T_amb')]:
        _p = _clean.get(_fam, {})
        _k = _p.get('iqr_k', 0.0) or 0.0
        if _k > 0:
            rows.append((f'{_lbl} IQR k (method 1)', _k)); _any_clean = True
        for _i, _d in enumerate(_p.get('dev_rounds', []), 1):
            if _d and _d > 0:
                rows.append((f'{_lbl} % dev round {_i} (method 2)', _d)); _any_clean = True
        _md = _p.get('max_diff', 0.0) or 0.0
        if _md > 0:
            rows.append((f'{_lbl} abs. max diff W/m2 (method 3)', _md)); _any_clean = True
    if not _any_clean:
        rows.append(('(no sensor cleaning configured)', ''))
    rows.append(('', ''))
    rows.append(('dt_h (hourly step)', dt_h))
    # Monthly minimum irradiance curves (07) when c5 or c6 are active.
    act = cfg.get('active_criteria', set())
    if 'c5' in act or 'c6' in act:
        poa_m = cfg.get('monthly_thr_poa', {})
        ghi_m = cfg.get('monthly_thr_ghi', {})
        rows.append(('', ''))
        rows.append(('--- Monthly minimum irradiance curves (07) ---', ''))
        rows.append(('month',        'POA_min / GHI_min'))
        for m in range(1, 13):
            rows.append((f'  month {m:02d}', f"{poa_m.get(m, 0.0):g} / {ghi_m.get(m, 0.0):g}"))
    return rows


def _order_proc_columns(cols_in, df=None, active=None):
    """Orders the scada_proc columns readably on sheet 03."""
    if df is not None:
        cols_in = [c for c in cols_in if c not in df.columns or not df[c].isna().all()]
    skip = {'plant_code', 'source_file', 'ingested_at', 'processed_at'}
    # The *_flag columns are not shown as their own column (the reason is annotated on
    # the _filt cell); they remain in the table and drive that annotation.
    cols_in = [c for c in cols_in if c not in skip and not c.endswith('_flag')]

    def _related(base):
        out = [base]
        for suf in ('_filt',):
            n = f'{base}{suf}'
            if n in cols_in:
                out.append(n)
        return out

    used = set()
    ordered = []
    if 'Date' in cols_in:
        ordered.append('Date'); used.add('Date')

    def _add_family(family, avg_filt_name=None):
        bases = sorted([c for c in cols_in
                        if c.upper().startswith(family)
                        and not c.endswith('_filt') and not c.endswith('_flag')
                        and c != avg_filt_name])
        for base in bases:
            for x in _related(base):
                if x not in used:
                    ordered.append(x); used.add(x)
        if avg_filt_name and avg_filt_name in cols_in and avg_filt_name not in used:
            ordered.append(avg_filt_name); used.add(avg_filt_name)

    _add_family('GHI_', 'GHI_avg_filt')
    _add_family('POA_', 'POA_avg_filt')
    _add_family('REF_', 'REF_avg_filt')
    _add_family('DHI_', 'DHI_avg_filt')
    _add_family('T_AMB', 'T_AMB_avg_filt')
    _add_family('WS_')
    _add_family('SETPOINT')
    _add_family('E_GRID')

    # Albedo
    for base in sorted([c for c in cols_in if c.upper().startswith('ALB_') and c != 'ALB_avg_filt']):
        if base not in used:
            ordered.append(base); used.add(base)
    if 'ALB_avg_filt' in cols_in and 'ALB_avg_filt' not in used:
        ordered.append('ALB_avg_filt'); used.add('ALB_avg_filt')

    _add_family('WA')
    _add_family('PF_')
    # Criteria and derived values. Only ACTIVE criteria are shown (active).
    _all_crit = ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
    crit_seq = _all_crit if active is None else [c for c in _all_crit if c in active]
    derived = ['poa_above_thr']
    if active is None or 'c5' in active:
        derived.append('daily_hours_above')
    if active is None or 'c6' in active:
        derived.append('daily_hours_above_ghi')
    derived.append('hours_aggregated')
    # INACTIVE criteria (and their hours/day) must not be shown nor enter all_valid.
    _skip = set(_all_crit) - set(crit_seq)
    if active is not None:
        if 'c5' not in active:
            _skip.add('daily_hours_above')
        if 'c6' not in active:
            _skip.add('daily_hours_above_ghi')
    for x in derived + crit_seq + ['all_valid']:
        if x in cols_in and x not in used and x not in _skip:
            ordered.append(x); used.add(x)

    for c in cols_in:
        if c not in used and c not in _skip:
            ordered.append(c); used.add(c)
    return ordered


def build_workbook(plant_code, period_str, cfg, dt_h,
                   scada_raw_df, scada_proc_df, pvsyst_df, outliers_df):
    """Hybrid mode:
        02/03/04/08 -> precomputed values (scada_raw, scada_proc, proc_pvsyst).
        05/06/07/00 -> Excel formulas over the preceding cells (recompute live).
    """
    wb = Workbook()
    wb.remove(wb.active)

    # ============================================================
    # 01_Contract_Config
    # ============================================================
    ws_cfg = wb.create_sheet('01_Contract_Config')
    _explain(ws_cfg,
        "Parameters read from plants_config.xlsx (01_Plants and 03_Criteria) and applied to the EPI calculation. "
        "The cells in column B are references used by the formulas in 05_Interval_Comparison, "
        "06_Daily_Summary and 00_Summary. Change a value and everything dependent "
        "recalculates live.",
        ncols=2)
    for i, (k, v) in enumerate(_config_layout(cfg, dt_h)):
        ws_cfg.cell(row=3 + i, column=1, value=k).font = Font(bold=k.startswith('---'))
        _put_val(ws_cfg, 3 + i, 2, v)
    ws_cfg.column_dimensions['A'].width = 32
    ws_cfg.column_dimensions['B'].width = 24

    # ============================================================
    # 00_Summary (first sheet; filled once the others exist)
    # ============================================================
    ws_sum = wb.create_sheet('00_Summary', 0)
    ws_sum.merge_cells('A1:F1')
    title = ws_sum.cell(row=1, column=1, value="Performance Ratio - Traceability")
    title.font = Font(name='Calibri', size=18, bold=True, color='1F4E79')
    title.alignment = Alignment(horizontal='left', vertical='center')
    ws_sum.row_dimensions[1].height = 32

    # ============================================================
    # 02_SCADA_Raw (raw values as-is)
    # ============================================================
    ws_raw = wb.create_sheet('02_SCADA_Raw')
    _explain(ws_raw,
        "SCADA readings exactly as received from the operator, renamed to the standard nomenclature "
        "(GHI_01, POA_01, REF_01, T_AMB_01, E_GRID_01, SETPOINT_01, WS_01) per "
        "plants_config.02_Sensors_Mapping. No transformations.",
        ncols=min(len(scada_raw_df.columns), 12))
    raw_cols = [c for c in scada_raw_df.columns if not scada_raw_df[c].isna().all()]
    scada_raw_df = scada_raw_df[raw_cols]  
    _hdr(ws_raw, raw_cols, 3)
    for r_idx, row_vals in enumerate(scada_raw_df.itertuples(index=False, name=None), start=4):
        for c_idx, v in enumerate(row_vals, 1):
            _put_val(ws_raw, r_idx, c_idx, v,
                     fmt=('yyyy-mm-dd hh:mm' if raw_cols[c_idx-1] == 'Date' else None))
    ws_raw.freeze_panes = 'B4'
    _autosize(ws_raw)
    if len(scada_raw_df) > 0:
        _apply_status_cf(ws_raw, 'A4', f'{get_column_letter(len(raw_cols))}{4 + len(scada_raw_df) - 1}')
        _add_table(ws_raw, 'tbl_raw', 3, 4 + len(scada_raw_df) - 1, len(raw_cols))

    # ============================================================
    # 03_SCADA_Processed (precomputed scada_proc values)
    # ============================================================
    ws_proc = wb.create_sheet('03_SCADA_Processed')
    _explain(ws_proc,
        "SCADA data after applying the pipeline in scada_proc.Notebook:\n"
        " 1. Anomaly detection per sensor (out_of_range / missing / dead / abrupt) -> colored cell.\n"
        " 2. Cross-sensor outlier on POA (4%). If the plant configures it, also GHI/DHI/T_amb (Plaza II: off).\n"
        " 3. Albedo = reflected / direct per 09_Albedo_Mapping (one ALB column per configured measure).\n"
        " 4. Active criteria (04_Criteria_Selection): c1-c4 avg_filt health and c7-c9 (live formulas);\n    c5/c6 valid hours/day POA/GHI (scada_proc values). all_valid = product of the active ones.\n"
        "Cells labeled 'NULL' (grey), 'OUT_OF_RANGE/DEAD/ABRUPT' (red), 'OUTLIER' (salmon) or "
        "'DISCARDED' (amber) indicate that value was discarded by the indicated filter.",
        ncols=14)

    headers = _order_proc_columns(list(scada_proc_df.columns), df=scada_proc_df,
                                  active=cfg.get('active_criteria'))
    _hdr(ws_proc, headers, 3)
    col_letter = {h: get_column_letter(i+1) for i, h in enumerate(headers)}

    data_start = 4
    n_rows = len(scada_proc_df)
    data_end = data_start + n_rows - 1

    proc_cols_in = list(scada_proc_df.columns)
    # Maps to distinguish NULL vs DISCARDED when writing values
    # 1) <sensor>_filt -> raw base (POA_01_filt -> POA_01)
    # Maps to distinguish NULL / DISCARDED / ANOMALY when writing values
    filt_to_raw = {}     # 'POA_01_filt' -> 'POA_01'
    raw_to_flag = {}     # 'POA_01' -> 'POA_01_flag'
    for h in headers:
        if h.endswith('_filt') and not h.endswith('_avg_filt'):
            base = h[:-len('_filt')]
            if base in proc_cols_in:
                filt_to_raw[h] = base
    # _flag is no longer a visible column: it is looked up in the full table to annotate
    # the raw/_filt cells with the reason (dead/missing/out_of_range/abrupt).
    for c in proc_cols_in:
        if c.endswith('_flag'):
            base = c[:-len('_flag')]
            if base in proc_cols_in:
                raw_to_flag[base] = c

    avg_filt_to_family = {
        'POA_avg_filt':   'POA_',
        'GHI_avg_filt':   'GHI_',
        'DHI_avg_filt':   'DHI_',
        'T_AMB_avg_filt': 'T_AMB',
        'ALB_avg_filt':   'ALB_',
    }
    avg_filt_to_raws = {}
    for avg_col, prefix in avg_filt_to_family.items():
        if avg_col not in proc_cols_in:
            continue
        raws = [c for c in proc_cols_in
                if c.upper().startswith(prefix.upper())
                and not c.endswith('_filt')
                and not c.endswith('_flag')
                and c != avg_col]
        avg_filt_to_raws[avg_col] = raws

    # avg_filt as a FORMULA = AVERAGE of the family's *_filt columns (linked).
    # (ALB_avg_filt is handled separately below.)
    avg_filt_to_filt = {}
    for avg_col, prefix in avg_filt_to_family.items():
        if avg_col not in headers or avg_col == 'ALB_avg_filt':
            continue
        fcols = [c for c in headers
                 if c.upper().startswith(prefix.upper())
                 and c.endswith('_filt') and c != avg_col]
        if fcols:
            avg_filt_to_filt[avg_col] = fcols

    # Precompute the column letters the formulas need
    ghi_col_letter      = next((col_letter[c] for c in headers
                                if c.upper().startswith('GHI_')
                                and not c.endswith('_filt')
                                and not c.endswith('_flag')), None)
    setpoint_col_letter = next((col_letter[c] for c in headers
                                if c.upper().startswith('SETPOINT')
                                and not c.endswith('_flag')), None)
    ws_col_letter       = next((col_letter[c] for c in headers
                                if c.upper().startswith('WS_')
                                and not c.endswith('_flag')), None)
    poa_avg_letter      = col_letter.get('POA_avg_filt')
    ghi_avg_letter      = col_letter.get('GHI_avg_filt')
    dhi_avg_letter      = col_letter.get('DHI_avg_filt')
    ref_avg_letter      = col_letter.get('REF_avg_filt')
    daily_hours_letter  = col_letter.get('daily_hours_above')
    ghi_daily_letter    = col_letter.get('daily_hours_above_ghi')
    poa_above_letter    = col_letter.get('poa_above_thr')
    wa_col_letter       = next((col_letter[c] for c in headers
                                if (c.upper().startswith('WA_') or c.upper() == 'WA')
                                and not c.endswith('_flag') and not c.endswith('_filt')), None)
    pf_col_letter       = next((col_letter[c] for c in headers
                                if c.upper().startswith('PF_')
                                and not c.endswith('_flag') and not c.endswith('_filt')), None)

    # Map each albedo measure -> (direct_sensor, reflected_sensor) from 09_Albedo_Mapping.
    # The live ALB formula divides reflected/direct on those two raw sensor columns.
    alb_to_pair = {}
    for _meas, _pair in cfg.get('albedo_mapping', {}).items():
        _direct, _refl = _pair
        if _meas in col_letter and _direct in col_letter and _refl in col_letter:
            alb_to_pair[_meas] = (_direct, _refl)
    alb_cols_in_headers = sorted([c for c in headers
                                  if c.upper().startswith('ALB_') and c != 'ALB_avg_filt'])

    for r_offset, row_vals in enumerate(scada_proc_df.itertuples(index=False, name=None)):
        r = data_start + r_offset
        for h in headers:
            if h not in proc_cols_in:
                continue
            idx = proc_cols_in.index(h)
            v = row_vals[idx]
            col_idx = headers.index(h) + 1
            fmt = 'yyyy-mm-dd hh:mm' if h == 'Date' else None

            # ---- Criteria as FORMULAS (not values) ------------------------
            # ---- ALB_XX as FORMULA: REF/base (config) with Plaza II convention ----
            #      den NaN or REF NaN -> "" ; den==0 or ratio<0 -> 0 ; otherwise -> REF/den
            if h in alb_to_pair:
                _den_col, _ref_col = alb_to_pair[h]
                _den_letter = col_letter[_den_col]
                _ref_letter = col_letter[_ref_col]
                f = (f'=IFERROR(IF(OR(NOT(ISNUMBER({_den_letter}{r})),NOT(ISNUMBER({_ref_letter}{r}))),"",'
                     f'IF({_den_letter}{r}=0,0,IF({_ref_letter}{r}/{_den_letter}{r}<0,0,'
                     f'{_ref_letter}{r}/{_den_letter}{r}))),"")')
                c = ws_proc.cell(row=r, column=col_idx, value=f)
                c.fill = _FORMULA_FILL
                c.number_format = '0.000'
                continue

            if h == 'ALB_avg_filt' and alb_cols_in_headers:
                # Mean of ALB_XX in [0, 1). Empty or out-of-range values are ignored.
                _first = col_letter[alb_cols_in_headers[0]]
                _last  = col_letter[alb_cols_in_headers[-1]]
                _alb_range = f'{_first}{r}:{_last}{r}'
                f = f'=IFERROR(AVERAGEIFS({_alb_range},{_alb_range},">0",{_alb_range},"<1"),0)'
                c = ws_proc.cell(row=r, column=col_idx, value=f)
                c.fill = _FORMULA_FILL
                c.number_format = '0.000'
                continue

            # ---- Criteria c1-c4 (avg_filt health) and c7-c9: LIVE FORMULAS ----
            #      (c5/c6 are not formulas: scada_proc values are written directly)
            if h == 'c1':
                f = (f'=IF(ISNUMBER({poa_avg_letter}{r}),IF({REF_C1_THR}<=0,1,'
                     f'IF({poa_avg_letter}{r}>={REF_C1_THR},1,0)),0)') if poa_avg_letter else '=0'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c2':
                f = (f'=IF(ISNUMBER({ghi_avg_letter}{r}),IF({REF_C2_THR}<=0,1,'
                     f'IF({ghi_avg_letter}{r}>={REF_C2_THR},1,0)),0)') if ghi_avg_letter else '=0'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c3':
                f = (f'=IF(ISNUMBER({dhi_avg_letter}{r}),IF({REF_C3_THR}<=0,1,'
                     f'IF({dhi_avg_letter}{r}>={REF_C3_THR},1,0)),0)') if dhi_avg_letter else '=0'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c4':
                f = (f'=IF(ISNUMBER({ref_avg_letter}{r}),IF({REF_C4_THR}<=0,1,'
                     f'IF({ref_avg_letter}{r}>={REF_C4_THR},1,0)),0)') if ref_avg_letter else '=0'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c5':
                c_obj = ws_proc.cell(row=r, column=col_idx, value=v)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c6':
                c_obj = ws_proc.cell(row=r, column=col_idx, value=v)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c7':
                # setpoint >= threshold -> the operator is NOT curtailing power
                f = (f'=IF(AND(ISNUMBER({setpoint_col_letter}{r}),'
                     f'{setpoint_col_letter}{r}>={REF_C7_SETPT}),1,0)') if setpoint_col_letter else '=1'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c8':
                # no Wind Alarm (WA != 1)
                f = f'=IF(N({wa_col_letter}{r})=1,0,1)' if wa_col_letter else '=1'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue
            if h == 'c9':
                # healthy PF (0.98 < PF < 1.0)
                f = (f'=IF(AND(ISNUMBER({pf_col_letter}{r}),{pf_col_letter}{r}>0.98,'
                     f'{pf_col_letter}{r}<1),1,0)') if pf_col_letter else '=1'
                c_obj = ws_proc.cell(row=r, column=col_idx, value=f)
                c_obj.fill = _FORMULA_FILL; c_obj.number_format = '[Color10]"✓";;""'
                continue

            if h == 'all_valid':
                # product of the ACTIVE criteria present on the sheet
                _present = [col_letter.get(ck) for ck in
                            ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']
                            if col_letter.get(ck)]
                f = ('=' + '*'.join(f'{l}{r}' for l in _present)) if _present else '=1'
                ws_proc.cell(row=r, column=col_idx, value=f).fill = _FORMULA_FILL
                continue

            if h in avg_filt_to_filt:
                # AVERAGE of the family's *_filt columns (ignores NULL/DISCARDED text).
                _cells = ','.join(f'{col_letter[fc]}{r}' for fc in avg_filt_to_filt[h])
                fobj = ws_proc.cell(row=r, column=col_idx, value=f'=IFERROR(AVERAGE({_cells}),"")')
                fobj.fill = _FORMULA_FILL
                fobj.number_format = '0.00'
                continue

            # ---- Standard value output ------------------------------------
            if h in filt_to_raw:
                base = filt_to_raw[h]
                raw_v = row_vals[proc_cols_in.index(base)]
                flag_v = row_vals[proc_cols_in.index(raw_to_flag[base])] if base in raw_to_flag else None
                _put_filt(ws_proc, r, col_idx, v, raw_v, flag_v=flag_v, fmt=fmt)
            elif h in avg_filt_to_raws:
                family_raws = [row_vals[proc_cols_in.index(c)] for c in avg_filt_to_raws[h]]
                _put_avg_filt(ws_proc, r, col_idx, v, family_raws, fmt=fmt)
            elif h in raw_to_flag:
                flag_v = row_vals[proc_cols_in.index(raw_to_flag[h])]
                _put_raw(ws_proc, r, col_idx, v, flag_v=flag_v, fmt=fmt)
            else:
                _put_val(ws_proc, r, col_idx, v, fmt=fmt)
    ws_proc.freeze_panes = 'B4'
    _autosize(ws_proc, max_w=18)
    if n_rows > 0:
        _apply_status_cf(ws_proc, f'A{data_start}', f'{get_column_letter(len(headers))}{data_end}')
        # Green background on all_valid when = 1
        if 'all_valid' in col_letter and n_rows > 0:
            from openpyxl.formatting.rule import CellIsRule
            av_letter = col_letter['all_valid']
            av_range = f'{av_letter}{data_start}:{av_letter}{data_end}'
            ws_proc.conditional_formatting.add(av_range,
                CellIsRule(operator='equal', formula=['1'],
                           fill=PatternFill('solid', start_color='C6EFCE', end_color='C6EFCE'),
                           font=Font(bold=True, color='006100')))

        _add_table(ws_proc, 'tbl_proc', 3, data_end, len(headers))
    proc_info = {
        'sheet': '03_SCADA_Processed',
        'data_start': data_start,
        'data_end': data_end,
        'col_letter': col_letter,
        'headers': headers,
    }

    # ============================================================
    # 04_PVsyst_Simulation
    # ============================================================
    ws_pv = wb.create_sheet('04_PVsyst_Simulation')
    _explain(ws_pv,
        "Output of the PVsyst simulation (proc_pvsyst), fed with the on-site measured meteo "
        "(via meteo_export). Expected-energy reference against which the real one is compared.",
        ncols=min(len(pvsyst_df.columns), 8))
    pv_cols = [c for c in pvsyst_df.columns if not pvsyst_df[c].isna().all()]
    pvsyst_df = pvsyst_df[pv_cols]
    _hdr(ws_pv, pv_cols, 3)
    for r_idx, row_vals in enumerate(pvsyst_df.itertuples(index=False, name=None), start=4):
        for c_idx, v in enumerate(row_vals, 1):
            _put_val(ws_pv, r_idx, c_idx, v,
                     fmt=('yyyy-mm-dd hh:mm' if pv_cols[c_idx-1] == 'Date' else None))
    ws_pv.freeze_panes = 'B4'
    _autosize(ws_pv)
    pv_col_letter = {h: get_column_letter(i+1) for i, h in enumerate(pv_cols)}
    pv_data_start = 4
    pv_data_end = 4 + len(pvsyst_df) - 1
    _add_table(ws_pv, 'tbl_pv', 3, pv_data_end, len(pv_cols))

    # ============================================================
    # 05_Interval_Comparison (formulas)
    # ============================================================
    ws_cmp = wb.create_sheet('05_Interval_Comparison')
    _explain(ws_cmp,
        "Comparison at the test resolution (1 h if PVsyst is hourly or aggregate_by_hour=True; "
        "otherwise the finest common one). The 15-min detail is in 03_SCADA_Processed." + chr(10) +
        " - Measured Energy (kWh)      = AVERAGE(E_GRID of the bucket in 03) * dt_h   (linked formula)" + chr(10) +
        " - Expected Energy (kWh)      = PVsyst E_Grid of that hour (link to 04) * dt_h   (linked formula)" + chr(10) +
        " - Valid                      = MIN(all_valid of the sub-intervals): the hour is valid only if all 4 quarters are" + chr(10) +
        " - Guaranteed Energy (kWh)    = Expected Energy * EPI * Degradation * Availability   (live formula)" + chr(10) +
        " - Delta (kWh)                = Measured Energy - Guaranteed Energy   (live formula)" + chr(10) +
        "Coloring: grey = not valid (Valid=0), green = Measured >= Guaranteed, red = Measured < Guaranteed.",
        ncols=8)
    cmp_headers = ['Date', 'Measured POA (W/m2)', 'Measured Energy (kWh)', 'Expected Energy PVsyst (kWh)',
                   'Guaranteed Energy (kWh)', 'Delta (kWh)', 'Valid', 'Expected POA (W/m2)', 'Hour']
    _hdr(ws_cmp, cmp_headers, 3)
    cmp_col = {h: get_column_letter(i+1) for i, h in enumerate(cmp_headers)}
    cmp_data_start = 4

    pv_egrid = 'E_Grid' if 'E_Grid' in pv_cols else next(
        (c for c in pv_cols if c != 'Date' and pd.api.types.is_numeric_dtype(pvsyst_df[c])), None
    )
    egrid_col = next((c for c in headers if c.upper().startswith('E_GRID')), None)

    # --- Aggregation at the test resolution (same rule as pr_runner) ---
    # Detail in 03 stays at 15 min; the comparison runs at the test
    # resolution: 1 h if PVsyst is hourly (or aggregate_by_hour=True). E_GRID and POA are
    # averaged, all_valid is aggregated with MIN (the hour counts only if all 4
    # quarters are valid), and expected energy uses the real hourly value -> delta matches pr_runner.
    _sc_diffs  = scada_proc_df['Date'].diff().dropna()
    _scada_res = _sc_diffs.mode().iloc[0] if not _sc_diffs.empty else pd.Timedelta(minutes=15)
    _pv_diffs  = pvsyst_df['Date'].diff().dropna()
    _pv_res    = _pv_diffs.mode().iloc[0] if not _pv_diffs.empty else pd.Timedelta(hours=1)
    if cfg.get('aggregate_by_hour'):
        _target = max(pd.Timedelta(hours=1), _scada_res, _pv_res)
    else:
        _target = max(_scada_res, _pv_res)
    _dt_h_cmp = _target.total_seconds() / 3600.0
    _freq = pd.tseries.frequencies.to_offset(_target)
    print(f"  Comparacion @ {_target} (dt_h={_dt_h_cmp})")

    # Formula linkage: each bucket (at the test resolution) is a
    # contiguous range of rows in 03 (scada_proc is sorted by Date, WITHOUT the T08
    # shift: the shift is added below, when building each comparison bucket).
    _date_l  = proc_info['col_letter']['Date']
    _egrid_l = proc_info['col_letter'].get(egrid_col) if egrid_col else None
    _poa_l   = proc_info['col_letter'].get('POA_avg_filt')
    _valid_l = proc_info['col_letter'].get('all_valid')
    _psheet  = proc_info['sheet']

    # Mismo time-shift que pr_runner: el desfase de T08 se le suma a PVsyst, que es
    # lo mismo que RESTARSELO al SCADA (serie nativa) antes de agrupar, redondeado
    # a su resolucion. proc_pvsyst va sin tocar, en su :00.
    _ts_min = scada_time_shift(scada_proc_df['Date'], plant_code, _scada_res)
    _bkeys  = (scada_proc_df['Date'] - pd.to_timedelta(_ts_min, unit='m')).dt.floor(_freq)

    # contiguous buckets: [bucket_key, row_03_start, row_03_end, shift_min]
    _buckets = []
    _prev = None
    for _off, _bk in enumerate(_bkeys):
        _pr = data_start + _off
        if _prev is None or _bk != _prev[0]:
            _buckets.append([_bk, _pr, _pr, int(_ts_min.iloc[_off])]); _prev = _buckets[-1]
        else:
            _prev[2] = _pr

    # --- Guaranteed-energy factor: base ($B$8..$B$10 in 01_Contract_Config) with
    #     optional per-period overrides (03_EPI_Factors), written as an editable
    #     helper table on this sheet (cols L..P) so the factor stays live per date. ---
    _epi_periods = cfg.get('epi_periods', [])
    _base_factor = f'{REF_EPI_GUAR}*{REF_DEGRADATION}*{REF_AVAILABILITY}'
    _guar_ranges = None
    if _epi_periods:
        _pc0 = 12
        for _j, _h in enumerate(['Period Start', 'Period End', 'EPI', 'Deg', 'Avail']):
            ws_cmp.cell(row=3, column=_pc0 + _j, value=_h).font = Font(bold=True)
        _p0 = 4
        for _i, _p in enumerate(_epi_periods):
            _rr = _p0 + _i
            _st = _p.get('start'); _en = _p.get('end')
            _st = pd.Timestamp('1900-01-01') if _st is None or pd.isna(_st) else pd.Timestamp(_st)
            _en = pd.Timestamp('2999-12-31') if _en is None or pd.isna(_en) else pd.Timestamp(_en)
            ws_cmp.cell(row=_rr, column=_pc0,     value=_st.to_pydatetime()).number_format = 'yyyy-mm-dd'
            ws_cmp.cell(row=_rr, column=_pc0 + 1, value=_en.to_pydatetime()).number_format = 'yyyy-mm-dd'
            ws_cmp.cell(row=_rr, column=_pc0 + 2, value=_p['epi']).number_format = '0.0000'
            ws_cmp.cell(row=_rr, column=_pc0 + 3, value=_p['deg']).number_format = '0.0000'
            ws_cmp.cell(row=_rr, column=_pc0 + 4, value=_p['avail']).number_format = '0.0000'
        _pn = _p0 + len(_epi_periods) - 1
        def _R(_c):
            _L = get_column_letter(_c)
            return f'${_L}${_p0}:${_L}${_pn}'
        _guar_ranges = (_R(_pc0), _R(_pc0 + 1), _R(_pc0 + 2), _R(_pc0 + 3), _R(_pc0 + 4))

    def _guar_formula(_ee, _a):
        if not _guar_ranges:
            return f'={_ee}*{_base_factor}'
        _Ls, _Le, _Lepi, _Ldeg, _Lav = _guar_ranges
        _match = f'(INT({_Ls})<=INT({_a}))*(INT({_a})<=INT({_Le}))'
        return (f'={_ee}*IF(SUMPRODUCT({_match})=0,{_base_factor},'
                f'SUMPRODUCT({_match}*{_Lepi}*{_Ldeg}*{_Lav}))')

    r_cmp = cmp_data_start
    for _bk, _r0, _r1, _sh in _buckets:
        _A = f"A{r_cmp}"
        # Date = fecha de 03 TAL CUAL: la columna A va en hora real, igual que las
        # fechas que escribe pr_runner. El time-shift T08 del tramo (en dias para
        # Excel) se le RESTA solo dentro de las busquedas contra PVsyst, que es
        # donde hace falta bajar a su escala.
        # La fecha de 03 se redondea HACIA ABAJO a la hora: el primer cuarto del
        # bucket cae a y cuarto en PL1 (sus 75 min) y pr_runner escribe la hora en
        # punto, asi que sin el redondeo la columna A no cuadraria con pr_results.
        # La busqueda contra PVsyst baja a su escala con el MISMO desfase redondeado.
        _sh_h  = int(round(_sh / 60.0)) * 60
        _sh_pv = _A if _sh_h == 0 else f'({_A}-{_sh_h}/1440)'
        ws_cmp.cell(row=r_cmp, column=1,
                    value=f"=FLOOR('{_psheet}'!{_date_l}{_r0},1/24)").number_format = 'yyyy-mm-dd hh:mm'
        # Measured POA = AVERAGE of the bucket in 03
        if _poa_l:
            ws_cmp.cell(row=r_cmp, column=2,
                        value=f"=IFERROR(AVERAGE('{_psheet}'!{_poa_l}{_r0}:{_poa_l}{_r1}),\"\")").number_format = '0.00'
        # Measured energy = AVERAGE(E_GRID of the bucket) * dt_h
        if _egrid_l:
            ws_cmp.cell(row=r_cmp, column=3,
                        value=f"=IFERROR(AVERAGE('{_psheet}'!{_egrid_l}{_r0}:{_egrid_l}{_r1}),0)*{_dt_h_cmp}").number_format = '0.000'
        else:
            ws_cmp.cell(row=r_cmp, column=3, value=0).number_format = '0.000'
        # Expected energy = PVsyst E_Grid for the hour (matched on $A - shift) * dt_h  (dynamic)
        if pv_egrid:
            ws_cmp.cell(row=r_cmp, column=4,
                        value=f"=SUMPRODUCT((INT(tbl_pv[Date]*24)=INT({_sh_pv}*24))*tbl_pv[{pv_egrid}])*{_dt_h_cmp}").number_format = '0.000'
        else:
            ws_cmp.cell(row=r_cmp, column=4, value=0).number_format = '0.000'
        # Guaranteed energy = expected * EPI * Deg * Avail (per-period factor by date)
        ee_ref = f'{cmp_col["Expected Energy PVsyst (kWh)"]}{r_cmp}'
        ws_cmp.cell(row=r_cmp, column=5,
                    value=_guar_formula(ee_ref, _A)).number_format = '0.000'
        # delta = measured - guaranteed
        em_ref = f'{cmp_col["Measured Energy (kWh)"]}{r_cmp}'
        eg_ref = f'{cmp_col["Guaranteed Energy (kWh)"]}{r_cmp}'
        ws_cmp.cell(row=r_cmp, column=6, value=f'={em_ref}-{eg_ref}').number_format = '0.000'
        # valid = MIN(all_valid of the bucket)
        if _valid_l:
            ws_cmp.cell(row=r_cmp, column=7,
                        value=f"=MIN('{_psheet}'!{_valid_l}{_r0}:{_valid_l}{_r1})")
        else:
            ws_cmp.cell(row=r_cmp, column=7, value=0)
        # Expected POA = PVsyst POA for the hour (matched on $A - shift)  (dynamic)
        if 'POA' in pv_col_letter:
            ws_cmp.cell(row=r_cmp, column=8,
                        value=f"=IFERROR(SUMPRODUCT((INT(tbl_pv[Date]*24)=INT({_sh_pv}*24))*tbl_pv[POA]),\"\")").number_format = '0.00'
        else:
            ws_cmp.cell(row=r_cmp, column=8, value="").number_format = '0.00'
        # Hour of day (hora real, $A) for the hourly profile
        ws_cmp.cell(row=r_cmp, column=9, value=f"=HOUR({_A})").number_format = '0'
        r_cmp += 1

    n_cmp = len(_buckets)
    ws_cmp.freeze_panes = 'B4'
    _autosize(ws_cmp)
    cmp_data_end = cmp_data_start + n_cmp - 1
    _add_table(ws_cmp, 'tbl_cmp', 3, cmp_data_end, len(cmp_headers))

    # --- Conditional formatting: NULL coloring + rows by status ---
    if n_rows > 0:
        last_col_letter = get_column_letter(len(cmp_headers))
        data_range = f'A{cmp_data_start}:{last_col_letter}{cmp_data_end}'

        # 1) Color cells containing 'NULL'/'DISCARDED' text, etc.
        _apply_status_cf(ws_cmp, f'A{cmp_data_start}', f'{last_col_letter}{cmp_data_end}')

        # 2) Row coloring by status (order matters: earlier rules win)
        from openpyxl.formatting.rule import FormulaRule
        # 2a) Grey if valid = 0 (interval not counted)
        ws_cmp.conditional_formatting.add(data_range,
            FormulaRule(formula=[f'$G{cmp_data_start}=0'],
                        fill=PatternFill('solid', start_color='E7E6E6', end_color='E7E6E6'),
                        stopIfTrue=True))
        # 2b) Green if measured >= guaranteed (valid interval, at or above guarantee)
        ws_cmp.conditional_formatting.add(data_range,
            FormulaRule(formula=[f'AND(ISNUMBER($C{cmp_data_start}),ISNUMBER($E{cmp_data_start}),$C{cmp_data_start}>=$E{cmp_data_start})'],
                        fill=PatternFill('solid', start_color='E2EFDA', end_color='E2EFDA'),
                        stopIfTrue=True))
        # 2c) Red if measured < guaranteed
        ws_cmp.conditional_formatting.add(data_range,
            FormulaRule(formula=[f'AND(ISNUMBER($C{cmp_data_start}),ISNUMBER($E{cmp_data_start}),$C{cmp_data_start}<$E{cmp_data_start})'],
                        fill=PatternFill('solid', start_color='FCE4D6', end_color='FCE4D6'),
                        stopIfTrue=True))

    cmp_info = {
        'sheet': '05_Interval_Comparison',
        'data_start': cmp_data_start,
        'data_end': cmp_data_end,
        'col_letter': cmp_col,
    }

    # ============================================================
    # 06_Daily_Summary (per-day SUMPRODUCT formulas)
    # ============================================================
    ws_d = wb.create_sheet('06_Daily_Summary')
    _explain(ws_d,
        "Aggregated per day (SUMPRODUCT over 05). The 'Valid' columns use only intervals with Valid=1. "
        "EPI = (Measured Valid - Guaranteed Valid)/Guaranteed Valid (delta % over the valid part). "
        "Measured/Expected POA = average ONLY over valid intervals.",
        ncols=12)
    d_headers = ['Date', 'Measured Energy (kWh)', 'Expected Energy (kWh)', 'Guaranteed Energy (kWh)', 'Delta (kWh)',
                 'Measured Energy Valid (kWh)', 'Guaranteed Energy Valid (kWh)', 'Delta Valid (kWh)', 'EPI (%)',
                 'Measured POA (W/m2)', 'Expected POA (W/m2)', 'Valid Intervals', 'EPI Positive (%)', 'EPI Negative (%)']
    _hdr(ws_d, d_headers, 3)
    unique_days = pd.Series(scada_proc_df['Date'].dt.date.unique()).sort_values().tolist()
    d_data_start = 4

    for i, day in enumerate(unique_days):
        r = d_data_start + i
        ws_d.cell(row=r, column=1, value=day).number_format = 'yyyy-mm-dd'
        dref = f"$A{r}*1"
        # Daily totals (all intervals)
        ws_d.cell(row=r, column=2, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[Measured Energy (kWh)])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=3, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[Expected Energy PVsyst (kWh)])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=4, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[Guaranteed Energy (kWh)])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=5, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[Delta (kWh)])').number_format = '#,##0.0'
        # Valid intervals only
        ws_d.cell(row=r, column=6, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[Valid]*tbl_cmp[Measured Energy (kWh)])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=7, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[Valid]*tbl_cmp[Guaranteed Energy (kWh)])').number_format = '#,##0.0'
        ws_d.cell(row=r, column=8, value=f'=F{r}-G{r}').number_format = '#,##0.0'
        ws_d.cell(row=r, column=9, value=f'=IFERROR((F{r}-G{r})/G{r}*100,"")').number_format = '0.00 "%"'
        # Measured and expected POA: average over VALID intervals only
        ws_d.cell(row=r, column=10,
                  value=f'=IFERROR(AVERAGEIFS(tbl_cmp[Measured POA (W/m2)],tbl_cmp[Date],">="&{dref},tbl_cmp[Date],"<"&({dref}+1),tbl_cmp[Valid],1),"")'
                  ).number_format = '0.00'
        ws_d.cell(row=r, column=11,
                  value=f'=IFERROR(AVERAGEIFS(tbl_cmp[Expected POA (W/m2)],tbl_cmp[Date],">="&{dref},tbl_cmp[Date],"<"&({dref}+1),tbl_cmp[Valid],1),"")'
                  ).number_format = '0.00'
        ws_d.cell(row=r, column=12, value=f'=SUMPRODUCT((INT(tbl_cmp[Date])={dref})*tbl_cmp[Valid])').number_format = '0'
        ws_d.cell(row=r, column=13, value=f'=IF(I{r}="","",IF(I{r}>=0,I{r},NA()))').number_format = '0.00'
        ws_d.cell(row=r, column=14, value=f'=IF(I{r}="","",IF(I{r}<0,I{r},NA()))').number_format = '0.00'

    ws_d.freeze_panes = 'A4'
    _autosize(ws_d, max_w=18)
    _last_d = d_data_start + len(unique_days) - 1
    _add_table(ws_d, 'tbl_daily', 3, _last_d, len(d_headers))

    # TOTALS row
    _tot = _last_d + 1
    ws_d.cell(row=_tot, column=1, value='TOTAL').font = Font(bold=True)
    for _ci, _cl in [(2, 'B'), (3, 'C'), (4, 'D'), (5, 'E'), (6, 'F'), (7, 'G'), (12, 'L')]:
        _cc = ws_d.cell(row=_tot, column=_ci, value=f'=SUM({_cl}{d_data_start}:{_cl}{_last_d})')
        _cc.number_format = '#,##0' if _ci == 12 else '#,##0.0'
        _cc.font = Font(bold=True)
    _cc = ws_d.cell(row=_tot, column=8, value=f'=F{_tot}-G{_tot}'); _cc.number_format = '#,##0.0'; _cc.font = Font(bold=True)
    _cc = ws_d.cell(row=_tot, column=9, value=f'=IFERROR((F{_tot}-G{_tot})/G{_tot}*100,"")'); _cc.number_format = '0.00 "%"'; _cc.font = Font(bold=True)

    # ============================================================
    # ============================================================
    # 00_Summary (final fill)
    # ============================================================
    em_sum_f = '=SUMIFS(tbl_cmp[Measured Energy (kWh)], tbl_cmp[Valid], 1)'
    ee_sum_f = '=SUMIFS(tbl_cmp[Expected Energy PVsyst (kWh)], tbl_cmp[Valid], 1)'
    eg_sum_f = '=SUMIFS(tbl_cmp[Guaranteed Energy (kWh)], tbl_cmp[Valid], 1)'

    # B-column positions (info_rows in 00_Summary, starting at row=3):
    #   row 3: Plant
    #   row 4: Period
    #   row 5: AC
    #   row 6: DC
    #   row 7: EPI
    #   row 8: Degradation
    #   row 9: Availability
    #   row 10: (empty)
    #   row 11: Measured energy
    #   row 12: Expected energy
    #   row 13: Guaranteed energy
    #   row 14: Delta kWh = B11 - B13
    #   row 15: Delta % = (B11-B13)/B13
    #   row 16: Result = IF(B11>=B13,"PASS","FAIL")
    info_rows = [
        ('Plant',                          f"{cfg['plant_name']}  ({plant_code})", None),
        ('Period',                         period_str, None),
        ('AC Capacity (kW)',               cfg['ac_capacity_kw'], '#,##0'),
        ('DC Capacity (kW)',               cfg['dc_capacity_kw'], '#,##0'),
        ('Guaranteed EPI',                 f'={REF_EPI_GUAR}', '0.0000'),
        ('Degradation factor',              f'={REF_DEGRADATION}', '0.0000'),
        ('Availability factor',             f'={REF_AVAILABILITY}', '0.0000'),
        ('', '', None),
        ('Measured Energy (kWh)',            em_sum_f, '#,##0.0'),
        ('Expected Energy PVsyst (kWh)',   ee_sum_f, '#,##0.0'),
        ('Guaranteed Energy (kWh)',       eg_sum_f, '#,##0.0'),
        ('Delta (kWh)',                     '=B11-B13', '#,##0.0'),
        ('Delta (%)',                       '=IFERROR((B11-B13)/B13*100,"")', '0.00 "%"'),
        ('Ratio (measured/guaranteed)',      '=IFERROR(B11/B13,"")', '0.0000'),
        ('Realized EPI',                   '=IFERROR(B11/B12,"")', '0.0000'),
        ('Guaranteed EPI (EPI*Deg*Avail)', f'={REF_EPI_GUAR}*{REF_DEGRADATION}*{REF_AVAILABILITY}', '0.0000'),
        ('Result',                       '=IF(B11>=B13,"PASS","FAIL")', None),
        ('', '', None),
        ('15-min intervals (raw)',       '=COUNTA(tbl_proc[Date])', '#,##0'),
        ('15-min intervals valid',       '=COUNTIF(tbl_proc[all_valid],1)', '#,##0'),
        ('Test intervals',              '=COUNTA(tbl_cmp[Date])', '#,##0'),
        ('Test intervals valid',      '=COUNTIF(tbl_cmp[Valid],1)', '#,##0'),
        ('Valid operating hours',        f'=COUNTIF(tbl_cmp[Valid],1)*{_dt_h_cmp}', '#,##0.0'),
    ]
    for i, (k, v, fmt) in enumerate(info_rows):
        r = 3 + i
        ws_sum.cell(row=r, column=1, value=k).font = Font(bold=bool(k))
        if v != '':
            cell = ws_sum.cell(row=r, column=2, value=v)
            if fmt:
                cell.number_format = fmt
            if k == 'Result':
                cell.font = _KPI_FONT
                from openpyxl.formatting.rule import FormulaRule
                ws_sum.conditional_formatting.add(f'B{r}',
                    FormulaRule(formula=[f'$B${r}="PASS"'], fill=_KPI_FILL_OK))
                ws_sum.conditional_formatting.add(f'B{r}',
                    FormulaRule(formula=[f'$B${r}="FAIL"'], fill=_KPI_FILL_KO))
    ws_sum.column_dimensions['A'].width = 32
    ws_sum.column_dimensions['B'].width = 28

    # ============================================================
    # 07_Chart (daily valid energy)
    # ============================================================
    from openpyxl.chart import LineChart, BarChart, Reference
    from openpyxl.chart.shapes import GraphicalProperties
    ws_g = wb.create_sheet('07_Chart')
    _explain(ws_g, "Valid energy and EPI. Hourly profile = average per hour of day over the date "
                   "range (edit From/To). EPI = (measured_v - guaranteed_v)/guaranteed_v. Bars: blue>=0, red<0.",
             ncols=10)
    _AZUL = '5B9BD5'
    _ROJO = 'D99694'
    if len(unique_days) > 0:
        _d06 = '06_Daily_Summary'
        cats_d = Reference(wb[_d06], min_col=1, min_row=d_data_start, max_row=_last_d)

        # 1) Daily lines: measured_valid (F=6) vs guaranteed_valid (G=7)
        ch1 = LineChart()
        ch1.title = "Daily valid energy - measured vs guaranteed"
        ch1.style = 12; ch1.y_axis.title = 'kWh'; ch1.x_axis.title = 'Date'
        ch1.height = 9; ch1.width = 26
        for _c in (6, 7):
            ch1.add_data(Reference(wb[_d06], min_col=_c, min_row=3, max_row=_last_d), titles_from_data=True)
        ch1.set_categories(cats_d)
        ws_g.add_chart(ch1, "B2")

        # 2) Daily EPI (blue/red bars, scale -100..100)
        ch2 = BarChart(); ch2.type = "col"
        ch2.title = "Daily EPI (%)"
        ch2.y_axis.title = '%'; ch2.x_axis.title = 'Date'
        ch2.height = 9; ch2.width = 26; ch2.overlap = 100; ch2.gapWidth = 60
        ch2.add_data(Reference(wb[_d06], min_col=13, min_row=3, max_row=_last_d), titles_from_data=True)
        ch2.add_data(Reference(wb[_d06], min_col=14, min_row=3, max_row=_last_d), titles_from_data=True)
        ch2.set_categories(cats_d)
        ch2.y_axis.scaling.min = -100; ch2.y_axis.scaling.max = 100
        ch2.x_axis.tickLblPos = "low"
        ch2.series[0].graphicalProperties = GraphicalProperties(solidFill=_AZUL)
        ch2.series[1].graphicalProperties = GraphicalProperties(solidFill=_ROJO)
        ws_g.add_chart(ch2, "B20")

        # ---- Date filter (editable) in T2/U2 and T3/U3 ----
        _dmin = pd.Timestamp(scada_proc_df['Date'].min()).to_pydatetime()
        _dmax = pd.Timestamp(scada_proc_df['Date'].max()).to_pydatetime()
        ws_g.cell(row=2, column=20, value='From:').font = Font(bold=True)
        ws_g.cell(row=2, column=21, value=_dmin).number_format = 'yyyy-mm-dd'
        ws_g.cell(row=3, column=20, value='To:').font = Font(bold=True)
        ws_g.cell(row=3, column=21, value=_dmax).number_format = 'yyyy-mm-dd'
        _DES = "$U$2"; _HAS = "$U$3"

        # ---- Hourly-profile table (0..23): average over the filter ----
        _hr0 = 5
        for _j, _hname in enumerate(['Hour', 'Measured Valid', 'Guaranteed Valid', 'EPI (%)', 'EPI Positive (%)', 'EPI Negative (%)']):
            ws_g.cell(row=_hr0, column=20 + _j, value=_hname).font = Font(bold=True)
        for _h in range(24):
            _r = _hr0 + 1 + _h
            ws_g.cell(row=_r, column=20, value=_h).number_format = '0'
            _flt = (f'tbl_cmp[Hour],T{_r},tbl_cmp[Valid],1,'
                    f'tbl_cmp[Date],">="&{_DES},tbl_cmp[Date],"<"&({_HAS}+1)')
            ws_g.cell(row=_r, column=21, value=f'=IFERROR(AVERAGEIFS(tbl_cmp[Measured Energy (kWh)],{_flt}),"")').number_format = '#,##0.000'
            ws_g.cell(row=_r, column=22, value=f'=IFERROR(AVERAGEIFS(tbl_cmp[Guaranteed Energy (kWh)],{_flt}),"")').number_format = '#,##0.000'
            ws_g.cell(row=_r, column=23, value=f'=IFERROR((U{_r}-V{_r})/V{_r}*100,"")').number_format = '0.00 "%"'
            ws_g.cell(row=_r, column=24, value=f'=IF(W{_r}="","",IF(W{_r}>=0,W{_r},NA()))').number_format = '0.00'
            ws_g.cell(row=_r, column=25, value=f'=IF(W{_r}="","",IF(W{_r}<0,W{_r},NA()))').number_format = '0.00'
        _hr_end = _hr0 + 24
        cats_h = Reference(ws_g, min_col=20, min_row=_hr0 + 1, max_row=_hr_end)

        # 3) Combined hourly profile: EPI bars (blue/red, primary axis) + energy lines (secondary axis)
        bar_h = BarChart(); bar_h.type = "col"
        bar_h.title = "Hourly profile (average of filtered days) - EPI and energy"
        bar_h.height = 11; bar_h.width = 28; bar_h.overlap = 100; bar_h.gapWidth = 60
        bar_h.add_data(Reference(ws_g, min_col=24, min_row=_hr0, max_row=_hr_end), titles_from_data=True)
        bar_h.add_data(Reference(ws_g, min_col=25, min_row=_hr0, max_row=_hr_end), titles_from_data=True)
        bar_h.set_categories(cats_h)
        bar_h.y_axis.title = 'EPI %'
        bar_h.y_axis.scaling.min = -100; bar_h.y_axis.scaling.max = 100
        bar_h.x_axis.title = 'Hour of day'
        bar_h.series[0].graphicalProperties = GraphicalProperties(solidFill=_AZUL)
        bar_h.series[1].graphicalProperties = GraphicalProperties(solidFill=_ROJO)

        line_h = LineChart()
        line_h.add_data(Reference(ws_g, min_col=21, min_row=_hr0, max_row=_hr_end), titles_from_data=True)
        line_h.add_data(Reference(ws_g, min_col=22, min_row=_hr0, max_row=_hr_end), titles_from_data=True)
        line_h.set_categories(cats_h)
        line_h.y_axis.axId = 200
        line_h.y_axis.title = 'kWh'
        line_h.y_axis.crosses = "max"
        bar_h += line_h
        ws_g.add_chart(bar_h, "B38")
    else:
        ws_g.cell(row=3, column=1, value="(no data to plot)").font = Font(italic=True, color='6F6F6F')

    # ============================================================
    # 08_Outliers (subset)
    # ============================================================
    ws_o = wb.create_sheet('08_Outliers')
    _explain(ws_o,
        "Only the intervals where at least one sensor was discarded by the cross-sensor "
        "cleaning (median-relative % rounds and/or absolute deviation) configured in 01_Contract_Config.",
        ncols=8)
    if outliers_df.empty:
        ws_o.cell(row=3, column=1, value="(No POA/GHI/DHI/REF outlier discards in this period)").font = Font(italic=True, color='6F6F6F')
    else:
        out_cols = list(outliers_df.columns)
        _hdr(ws_o, out_cols, 3)
        for r_idx, row_vals in enumerate(outliers_df.itertuples(index=False, name=None), start=4):
            for c_idx, v in enumerate(row_vals, 1):
                _put_val(ws_o, r_idx, c_idx, v,
                         fmt=('yyyy-mm-dd hh:mm' if out_cols[c_idx-1] == 'Date' else None))
        ws_o.freeze_panes = 'B4'
        _autosize(ws_o)
        _apply_status_cf(ws_o, 'A4', f'{get_column_letter(len(out_cols))}{4 + len(outliers_df) - 1}')

    # ============================================================
    # Final sheet order
    # ============================================================
    new_order = ['00_Summary', '01_Contract_Config', '02_SCADA_Raw',
                 '03_SCADA_Processed', '04_PVsyst_Simulation',
                 '05_Interval_Comparison', '06_Daily_Summary',
                 '07_Chart', '08_Outliers']
    wb._sheets = [wb[n] for n in new_order if n in wb.sheetnames]

    return wb


In [ ]:
def _norm_plant_codes(v):
    """Normalizes plant_codes to a list (one or many). None/'' = all.
    Accepts a list/tuple, JSON array, or comma-separated string."""
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    try:
        import json as _json
        parsed = _json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, TypeError):
        pass
    return [p.strip() for p in s.split(',') if p.strip()]


plant_list = _norm_plant_codes(plant_codes)
if not plant_list:
    plant_list = [r['plant_code'] for r in spark.sql(
        "SELECT DISTINCT plant_code FROM scada_proc ORDER BY plant_code").collect()]

# Filter by the 'traceability_excel' column of 01_Plants (unless force_export).
skipped_plants = []
if force_export:
    print("force_export=True: se ignora la columna traceability_excel de plants_config.")
else:
    _plants_cfg = _read_sheet('01_Plants', 'T01_plants')
    _enabled = {str(r['plant_code']).strip()
                for _, r in _plants_cfg.iterrows()
                if _bool(r.get('traceability_excel'), False)}
    skipped_plants = [p for p in plant_list if p not in _enabled]
    plant_list = [p for p in plant_list if p in _enabled]
    if skipped_plants:
        print(f"Omitidas (traceability_excel=FALSE en plants_config): {skipped_plants}")
print(f"Plantas a procesar ({len(plant_list)}): {plant_list}")


def process_plant(plant_code):
    _ps = period_start
    _pe = period_end

    # ---------------------------------------------------------------------------
    # Pipeline: read data, build the workbook, upload it
    # ---------------------------------------------------------------------------
    print(f"Cargando configuración de {plant_code}...")
    cfg = load_plant_config(plant_code)

    # Determine the period from the INTERSECTION of scada_proc and proc_pvsyst
    # (traceability is impossible where there is no PVsyst simulation to compare against)
    sc_row = spark.sql(f"""
        SELECT MIN(Date) AS mn, MAX(Date) AS mx
        FROM scada_proc
        WHERE plant_code = '{plant_code}'
    """).first()
    pv_row = spark.sql(f"""
        SELECT MIN(Date) AS mn, MAX(Date) AS mx
        FROM proc_pvsyst
        WHERE plant_code = '{plant_code}'
    """).first()

    if sc_row['mn'] is None:
        raise ValueError(f"No hay datos en scada_proc para {plant_code} — ejecuta primero scada_ingest + scada_proc.")
    if pv_row['mn'] is None:
        raise ValueError(
            f"No hay datos en proc_pvsyst para {plant_code}. "
            "Ejecuta antes el PVsyst CLI con la salida de meteo_export y luego csv_ingest."
        )

    print(f"  scada_proc  disponible: {sc_row['mn']} -> {sc_row['mx']}")
    print(f"  proc_pvsyst disponible: {pv_row['mn']} -> {pv_row['mx']}")

    intersect_start = max(pd.Timestamp(sc_row['mn']), pd.Timestamp(pv_row['mn']))
    intersect_end   = min(pd.Timestamp(sc_row['mx']), pd.Timestamp(pv_row['mx']))
    if intersect_end < intersect_start:
        raise ValueError(
            f"Sin solape entre scada_proc y proc_pvsyst para {plant_code}: "
            f"scada=[{sc_row['mn']}..{sc_row['mx']}], pvsyst=[{pv_row['mn']}..{pv_row['mx']}]"
        )

    # Normalize whatever the user entered (accepts ISO YYYY-MM-DD; empty = not set)
    _req_start = pd.Timestamp(_ps) if str(_ps).strip() else None
    _req_end   = pd.Timestamp(_pe)   if str(_pe).strip()   else None

    if _req_start is not None or _req_end is not None:
        # PRIORITY: if _ps/_pe are set, they override last_month_only
        _ps_dt = _req_start if _req_start is not None else intersect_start
        _pe_dt   = _req_end   if _req_end   is not None else intersect_end

        # Check availability and clip to the real data overlap
        if _ps_dt < intersect_start:
            print(f"  AVISO: inicio pedido {_ps_dt} < disponible {intersect_start}; se recorta al disponible.")
            _ps_dt = intersect_start
        if _pe_dt > intersect_end:
            print(f"  AVISO: fin pedido {_pe_dt} > disponible {intersect_end}; se recorta al disponible.")
            _pe_dt = intersect_end
        if _pe_dt < _ps_dt:
            raise ValueError(
                f"El periodo pedido [{_ps} .. {_pe}] no se solapa con los datos "
                f"disponibles [{intersect_start} .. {intersect_end}]."
            )

        _ps = _ps_dt.isoformat()
        _pe   = _pe_dt.isoformat()
        print(f"Modo 'periodo manual' (_ps/_pe tienen prioridad): {_ps} -> {_pe}")
    elif last_months_only:
        _pe_dt   = intersect_end
        _ps_dt = max(intersect_start, _pe_dt - timedelta(days=num_months * 30))
        _ps = _ps_dt.isoformat()
        _pe   = _pe_dt.isoformat()
        print(f"Modo 'últimos {num_months} meses' (sobre la intersección de datos): {_ps} -> {_pe}")
    else:
        _ps = intersect_start.isoformat()
        _pe   = intersect_end.isoformat()
        print(f"Modo 'rango completo' (toda la intersección de datos): {_ps} -> {_pe}")

    _ps_f = pd.Timestamp(_ps).strftime('%Y%m%d')
    _pe_f   = pd.Timestamp(_pe).strftime('%Y%m%d')
    period_str = f"{pd.Timestamp(_ps).strftime('%d/%m/%Y')} a {pd.Timestamp(_pe).strftime('%d/%m/%Y')}"
    print(f"Periodo final: {period_str}")

    # --- Early check: confirm the 3 tables have data in the period BEFORE the heavy work ---
    print("\nComprobando disponibilidad de datos en el periodo...")
    for tbl in ('scada_raw', 'scada_proc', 'proc_pvsyst'):
        n = spark.sql(f"""
            SELECT COUNT(*) AS n
            FROM {tbl}
            WHERE plant_code = '{plant_code}'
              AND Date BETWEEN TIMESTAMP'{_ps}' AND TIMESTAMP'{_pe}'
        """).first()['n']
        print(f"  {tbl:12s}: {n:,} filas")
        if n == 0:
            # Helpful hint: what global range does that table have for the plant?
            rg = spark.sql(f"""
                SELECT MIN(Date) AS mn, MAX(Date) AS mx
                FROM {tbl}
                WHERE plant_code = '{plant_code}'
            """).first()
            if rg['mn'] is None:
                extra = "la tabla está completamente vacía para esta planta."
                if tbl == 'proc_pvsyst':
                    extra += " Ejecuta el PVsyst CLI con la salida de meteo_export y luego csv_ingest."
            else:
                extra = f"la tabla tiene datos solo entre {rg['mn']} y {rg['mx']}."
            raise ValueError(
                f"{tbl} vacío para {plant_code} en el periodo {_ps} -> {_pe}. "
                f"Pista: {extra}"
            )


    def _read_table(table, plant_code, _ps, _pe):
        sql = f"""
            SELECT *
            FROM {table}
            WHERE plant_code = '{plant_code}'
              AND Date BETWEEN TIMESTAMP'{_ps}' AND TIMESTAMP'{_pe}'
            ORDER BY Date
        """
        return spark.sql(sql).toPandas()


    print("\nLeyendo datos...")
    scada_raw  = _read_table('scada_raw',  plant_code, _ps, _pe).drop(columns=['source_file', 'ingested_at', 'plant_code'], errors='ignore')
    scada_proc = _read_table('scada_proc', plant_code, _ps, _pe).drop(columns=['source_file', 'ingested_at', 'processed_at', 'plant_code'], errors='ignore')
    pvsyst     = _read_table('proc_pvsyst', plant_code, _ps, _pe).drop(columns=['source_file', 'ingested_at', 'plant_code'], errors='ignore')

    print(f"  scada_raw : {len(scada_raw):,} filas")
    print(f"  scada_proc: {len(scada_proc):,} filas")
    print(f"  pvsyst    : {len(pvsyst):,} filas")

    # Detect the SCADA dt_h
    diffs = scada_raw['Date'].diff().dropna()
    dt_h = float(diffs.mode().iloc[0].total_seconds() / 3600) if not diffs.empty else 0.25
    print(f"  dt_h      : {dt_h} horas/intervalo")
    # ---------------------------------------------------------------------------
    # Outlier subset (for sheet 08; the rest is formula-driven)
    # ---------------------------------------------------------------------------
    # Outliers from all sensor families (POA, GHI, DHI, REF).
    # All-null columns are ignored (sensor not installed at this plant).
    _OUTLIER_FAMILIES = [
        ('POA_', 'POA_avg_filt'),
        ('GHI_', 'GHI_avg_filt'),
        ('DHI_', 'DHI_avg_filt'),
        ('REF_', 'REF_avg_filt'),
    ]
    _all_out_idx  = set()
    _all_keep_cols = []
    for _pfx, _avg in _OUTLIER_FAMILIES:
        _raw  = sorted([c for c in scada_proc.columns
                        if c.upper().startswith(_pfx)
                        and not c.endswith('_filt') and not c.endswith('_flag')
                        and c != _avg])
        _filt = sorted([c for c in scada_proc.columns
                        if c.upper().startswith(_pfx)
                        and c.endswith('_filt') and c != _avg])
        # Drop fully-null columns (sensor does not exist at this plant)
        _raw  = [c for c in _raw  if not scada_proc[c].isna().all()]
        _filt = [c for c in _filt if not scada_proc[c].isna().all()]
        if not _raw or not _filt:
            continue
        _raw_num  = scada_proc[_raw].apply(pd.to_numeric, errors='coerce')
        _filt_num = scada_proc[_filt].apply(pd.to_numeric, errors='coerce')
        _mask = _filt_num.isna().any(axis=1) & (_raw_num.fillna(0).abs().sum(axis=1) > 0)
        _all_out_idx.update(scada_proc.index[_mask].tolist())
        _all_keep_cols += _raw + _filt
        if _avg in scada_proc.columns and not scada_proc[_avg].isna().all():
            _all_keep_cols.append(_avg)
    if _all_out_idx:
        outliers_df = scada_proc.loc[sorted(_all_out_idx)].copy()
        _seen = set(); _keep = ['Date']
        for _c in _all_keep_cols:
            if _c not in _seen and _c in outliers_df.columns:
                _keep.append(_c); _seen.add(_c)
        outliers_df = outliers_df[_keep]
    else:
        outliers_df = pd.DataFrame()

    print(f"  Filas con descartes (POA/GHI/DHI/REF): {len(outliers_df):,}")
    # ---------------------------------------------------------------------------
    # Build the Excel and upload it
    # ---------------------------------------------------------------------------
    wb = build_workbook(plant_code, period_str, cfg, dt_h,
                        scada_raw_df=scada_raw,
                        scada_proc_df=scada_proc,
                        pvsyst_df=pvsyst,
                        outliers_df=outliers_df)

    buf = io.BytesIO()
    wb.save(buf)
    content_bytes = buf.getvalue()
    print(f"\nExcel generado: {len(content_bytes)/1024:.1f} KB")

    # File name
    fname = f"{plant_code}_PR_Traceability_{_ps_f}-{_pe_f}.xlsx"

    # 1) Local audit
    audit_dir = f"{LAKEHOUSE_FILES}/{local_audit_root.strip('/')}/{plant_code}"
    Path(audit_dir).mkdir(parents=True, exist_ok=True)
    audit_path = f"{audit_dir}/{fname}"
    with open(audit_path, 'wb') as f:
        f.write(content_bytes)
    print(f"  audit local: {audit_path.replace('/lakehouse/default/', '')}")

    # 2) Upload to SharePoint via Graph
    sharepoint_url = None
    try:
        print("\nAutenticando contra Microsoft Graph...")
        _token  = get_graph_token()
        _site_id, _drive_id = resolve_site_and_drive(_token)
        dest_relpath = f"{sp_traceability_path.strip('/')}/{plant_code}/{fname}"
        info = upload_to_sharepoint(_token, _site_id, _drive_id, dest_relpath, content_bytes)
        sharepoint_url = info.get('webUrl')
        print(f"  SharePoint:  {sharepoint_url}")
    except Exception as _e:
        print(f"  WARN: upload a SharePoint fallido (se conserva la copia local): {_e}")

    return {
        'plant_code':      plant_code,
        'period_start':    _ps_f,
        'period_end':      _pe_f,
        'rows_scada_raw':  int(len(scada_raw)),
        'rows_scada_proc': int(len(scada_proc)),
        'rows_pvsyst':     int(len(pvsyst)),
        'rows_outliers':   int(len(outliers_df)),
        'local_audit':     audit_path.replace('/lakehouse/default/', ''),
        'sharepoint':      sharepoint_url,
    }



In [ ]:
import notebookutils
results = []
for _pc in plant_list:
    try:
        results.append({'status': 'ok', **process_plant(_pc)})
    except Exception as _e:
        results.append({'plant_code': _pc, 'status': 'failed', 'error': str(_e)[:300]})
        print(f"  FALLO {_pc}: {_e}")

_summary = {
    'plants_processed': len(results),
    'ok':     sum(1 for r in results if r.get('status') == 'ok'),
    'failed': sum(1 for r in results if r.get('status') == 'failed'),
    'skipped': skipped_plants,
    'results': results,
}
print('\n' + json.dumps(_summary, indent=2, default=str))
notebookutils.notebook.exit(json.dumps(_summary, default=str))